# 19 Differential Abundance and symptom-related gluten avoidance

In the first part of this notebook, we use ANCOM-BC2 to test whether fungal genus abundance differs between participants who report avoiding gluten because it makes them feel bad and those who report no gluten-related issue (`gluten_sample`). We use the unrarefied genus count table from notebook 14 and keep the same samples for the unadjusted and adjusted models. The adjusted model accounts for age and sex, as in notebook 13.


## 1. Setup

### 1.1 Imports

In [1]:
import pandas as pd
import qiime2 as q2
from qiime2 import Artifact, Metadata

In [4]:
metadata = "../data/raw/Final_metadata_with_GHSL_exclude_other_sex.tsv"
in_dir = "../data/processed_ignored/14_differential_abundance_setup"
out_dir = "../data/processed_ignored/20_differential_abundance_gluten"
vis_dir = "../data/processed/20_differential_abundance_gluten"

genus_table = f"{in_dir}/table_fungi_l6.qza"
gluten_table = f"{out_dir}/table_fungi_l6_gluten.qza"

gluten_unadjusted_dir = f"{vis_dir}/unadjusted_ancombc2" 
gluten_adjusted_dir = f"{vis_dir}/adjusted_ancombc2"

gluten_unadjusted_qza = f"{out_dir}/ancombc2_gluten_unadjusted.qza"
gluten_adjusted_qza = f"{out_dir}/ancombc2_gluten_adjusted.qza"

gluten_unadjusted_tabulate = f"{gluten_unadjusted_dir}/tabulate_gluten_unadjusted.qzv"
gluten_adjusted_tabulate = f"{gluten_adjusted_dir}/tabulate_gluten_adjusted.qzv"

gluten_unadjusted_barplot = f"{gluten_unadjusted_dir}/barplot_gluten_unadjusted.qzv"
gluten_adjusted_barplot = f"{gluten_adjusted_dir}/barplot_gluten_adjusted.qzv"

!mkdir -p "$out_dir" "$vis_dir"
!mkdir -p "$out_dir" "$vis_dir"
!mkdir -p "$gluten_unadjusted_dir" "$gluten_adjusted_dir"


### 1.3 Load inputs

We load the genus count table from notebook 14 and the cleaned sample metadata used in notebook 09.

In [6]:
genus_artifact = Artifact.load(genus_table)
sample_metadata = Metadata.load(metadata)

print("Table type:", genus_artifact.type)
print("Samples in metadata:", sample_metadata.id_count)

Table type: FeatureTable[Frequency]
Samples in metadata: 149


## 2. Select samples

### 2.1 Keep complete cases

We keep samples with settlement type, age, and diet information so both models use the same samples.

In [8]:
!qiime feature-table filter-samples \
    --i-table "$genus_table" \
    --m-metadata-file "$metadata" \
    --p-where "[age_years_sample] IS NOT NULL AND [sex_sample] IS NOT NULL AND [gluten_sample] IS NOT NULL" \
    --o-filtered-table "$gluten_table"

Saved FeatureTable[Frequency] to: ../data/processed_ignored/20_differential_abundance_gluten/table_fungi_l6_gluten.qza


### 2.2 Summarize selected samples

We summarize the numeric age values after filtering for complete settlement, sex, and age information.

In [9]:
gluten_counts = Artifact.load(gluten_table).view(pd.DataFrame)
selected_metadata = sample_metadata.to_dataframe().loc[gluten_counts.index]

print("Selected samples:", len(selected_metadata))
display(selected_metadata["gluten_sample"].value_counts())

Selected samples: 140


gluten_sample
No                                                                               110
I do not eat gluten because it makes me feel bad                                  25
I was diagnosed with gluten allergy (anti-gluten IgG), but not celiac disease      4
I was diagnosed with celiac disease                                                1
Name: count, dtype: int64

## 3. Run ANCOMBC-2

We use ANCOM-BC2 because some fungal genera were detected in only a few samples. When a genus has many zero counts, a result can depend on the small value used in place of zero. ANCOM-BC2 checks this by trying several small values and seeing whether the result stays significant. 

We add `--p-diff-robust` to both models to see which results stay significant when ANCOM-BC2 tries different small values for the zero counts.

Since the metadata labels "I was diagnosed with gluten allergy (anti-gluten IgG), but not celiac disease" and "I was diagnosed with celiac disease" are just represented by 4 and 1 sample(s) in the survey, respectively, their differential abundances are not being analysed here. Instead, we compare the group "I do not eat gluten because it makes me feel bad" to the reference group "No". Participants belonging to this group consume gluten and show no symptoms associated with a gluten intolerance or allergy. We can therefore consider it as the "control group" here. 

In [12]:
!qiime composition ancombc2 --help

Usage: qiime composition ancombc2 [OPTIONS]

  Calls the `ancombc2` function of the ANCOMBC software package. See the
  ANCOM-BC2 publication and source code for details.

  Sensitivity Analysis for Pseudo-Count Addition: To assess robustness, a
  series of pseudo-counts (0.01 to 0.5, in 0.01 increments) is added to zero
  counts, and a linear model is fitted to the bias-corrected log-abundance
  data for each. A sensitivity score is calculated as the proportion of
  p-values above the specified alpha. Taxa with consistent significance across
  all pseudo-counts, and agreement with complete-data results, are flagged as
  robust (diff_robust = TRUE). Note: Results depend on the selected alpha
  value.

Inputs:
  --i-table ARTIFACT FeatureTable[Frequency]
                         The feature table to be used for ANCOM-BC2
                         computation.                               [required]
Parameters:
  --m-metadata-file METADATA...
    (multiple            The per-sample metad

### 3.1 Unadjusted ANCOM-BC2

In [13]:
!qiime composition ancombc2 \
    --i-table "$gluten_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "gluten_sample" \
    --p-reference-levels gluten_sample::No \
    --p-diff-robust \
    --o-ancombc2-output "$gluten_unadjusted_qza"

Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/20_differential_abundance_gluten/ancombc2_gluten_unadjusted.qza


In [14]:
!qiime composition tabulate \
    --i-data "$gluten_unadjusted_qza" \
    --o-visualization "$gluten_unadjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$gluten_unadjusted_qza" \
    --o-visualization "$gluten_unadjusted_barplot"


Saved Visualization to: ../data/processed/20_differential_abundance_gluten/unadjusted_ancombc2/tabulate_gluten_unadjusted.qzv
Saved Visualization to: ../data/processed/20_differential_abundance_gluten/unadjusted_ancombc2/barplot_gluten_unadjusted.qzv


In [18]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(gluten_unadjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["I do not eat gluten because it makes me feel bad"]:
    contrast = f"gluten_sample::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. No symptom-related gluten avoidance  — {len(significant)} taxa with q < 0.05")
    display(significant)

I do not eat gluten because it makes me feel bad vs. No symptom-related gluten avoidance  — 3 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Oligophagozyma,2.493728,0.001556,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Saccharomyces,1.990475,0.004828,False
k__Fungi;p__Basidiomycota;c__Tremellomycetes;o__Filobasidiales;f__Filobasidiaceae;g__Naganishia,-1.769019,0.032100,False


#### Unadjusted results

**I do not eat gluten because it makes me feel bad vs. No symptom-related gluten avoidance**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Oligophagozyma* | +2.494 | 0.001556 | False |
| *Saccharomyces* | +1.990 | 0.004828 | False |
| *Naganishia* | −1.769 | 0.032100 | False |



Three genera had q < 0.05. Compared with the reference group, participants who reported avoiding gluten because it made them feel bad had higher estimated abundance of *Oligophagozyma* and *Saccharomyces* and lower estimated abundance of *Naganishia*. None of the three associations passed the sensitivity check for zero counts.

### 3.2 Adjusted ANCOM-BC2

We observe gluten-consumption-associated shifts in mycobial abundance after accounting for age and diet. We use the same 140 samples as in the unadjusted model.

In [19]:
!qiime composition ancombc2 \
    --i-table "$gluten_table" \
    --m-metadata-file "$metadata" \
    --p-fixed-effects-formula "age_years_sample + sex_sample + gluten_sample" \
    --p-reference-levels gluten_sample::No \
    --p-diff-robust \
    --o-ancombc2-output "$gluten_adjusted_qza"

!qiime composition tabulate \
    --i-data "$gluten_adjusted_qza" \
    --o-visualization "$gluten_adjusted_tabulate"

!qiime composition da-barplot \
    --i-data "$gluten_adjusted_qza" \
    --o-visualization "$gluten_adjusted_barplot"


Saved FeatureData[ANCOMBC2Output] to: ../data/processed_ignored/20_differential_abundance_gluten/ancombc2_gluten_adjusted.qza
Saved Visualization to: ../data/processed/20_differential_abundance_gluten/adjusted_ancombc2/tabulate_gluten_adjusted.qzv
Saved Visualization to: ../data/processed/20_differential_abundance_gluten/adjusted_ancombc2/barplot_gluten_adjusted.qzv


In [20]:
from zipfile import ZipFile
from IPython.display import display

with ZipFile(gluten_adjusted_qza) as artifact:
    data_dir = artifact.namelist()[0].split("/")[0] + "/data"
    tables = {}

    for name in ["q", "lfc", "diff_robust"]:
        with artifact.open(f"{data_dir}/{name}.jsonl") as file:
            file.readline()  # Skip the JSONL header
            tables[name] = pd.read_json(file, lines=True).set_index("taxon")

for group in ["I do not eat gluten because it makes me feel bad"]:
    contrast = f"gluten_sample::{group}"

    results = pd.DataFrame({
        "log_fold_change": tables["lfc"][contrast],
        "q_value": tables["q"][contrast],
        "diff_robust": tables["diff_robust"][contrast],
    })

    significant = results.loc[results["q_value"] < 0.05].sort_values("q_value")
    print(f"{group} vs. No symptom-related gluten avoidance  — {len(significant)} taxa with q < 0.05")
    display(significant)

I do not eat gluten because it makes me feel bad vs. No symptom-related gluten avoidance  — 5 taxa with q < 0.05


,log_fold_change,q_value,diff_robust
taxon,,,
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Oligophagozyma,3.333203,0.000051,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Metschnikowiaceae;g__Clavispora,2.771398,0.000690,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Saccharomyces,2.006654,0.001317,False
k__Fungi;p__Basidiomycota;c__Tremellomycetes;o__Filobasidiales;f__Filobasidiaceae;g__Naganishia,-2.275895,0.002627,False
k__Fungi;p__Ascomycota;c__Saccharomycetes;o__Saccharomycetales;f__Saccharomycetaceae;g__Kluyveromyces,1.973608,0.009672,False


#### Adjusted results

**I do not eat gluten because it makes me feel bad vs. No symptom-related gluten avoidance**

| Taxon | LFC | q-value | diff_robust |
|---|---:|---:|:---:|
| *Oligophagozyma* | +3.333 | 0.000051 | False |
| *Clavispora* | +2.771 | 0.000690 | False |
| *Saccharomyces* | +2.007 | 0.001317 | False |
| *Naganishia* | −2.276 | 0.002627 | False |
| *Kluyveromyces* | +1.974 | 0.009672 | False |



Five genera had q < 0.05 after adjustment. In both models, participants who reported avoiding gluten because it made them feel bad had higher estimated abundance of *Oligophagozyma* and *Saccharomyces* and lower estimated abundance of *Naganishia* than the reference group. *Clavispora* and *Kluyveromyces* also had higher estimated abundance in this group and reached q < 0.05 only after adjustment. None of the five adjusted associations passed the sensitivity check for zero counts.

## 4. Discussion

Before and after adjustment, participants who reported avoiding gluten because it made them feel bad had higher estimated abundance of *Oligophagozyma* and *Saccharomyces* and lower estimated abundance of *Naganishia* than the reference group. *Clavispora* and *Kluyveromyces* also showed higher estimated abundance, but reached q < 0.05 only after adjustment. None of these associations remained significant across all tested pseudocounts, so the evidence for differences related to symptom-driven gluten avoidance remains limited.